In [ ]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType
from pyspark.ml.feature import Imputer
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

In [ ]:
JAR_PATH_1 = os.path.abspath("./jars/hadoop-aws-3.4.0.jar")
JAR_PATH_2 = os.path.abspath("./jars/aws-sdk-s3-2.29.52.jar")

JARS_LIST = f"{JAR_PATH_1},{JAR_PATH_2}"

In [ ]:
spark = (
    SparkSession.builder.appName("analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.jars.repositories", "https://repo1.maven.org/maven2/")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

# Customer Related Analysis 

active customers over time

In [ ]:


dataframes["customers"] = dataframes["customers"].withColumn(
    "account_created_date",
    F.to_date("account_created_at")
)

def active_customers_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.filter(F.col("is_active") == True)
          .groupBy(*group_cols)
          .agg(F.countDistinct("customer_id").alias("active_customers"))
          .orderBy(*group_cols)
    )

daily_active   = active_customers_over_time(dataframes["customers"], "day")
weekly_active  = active_customers_over_time(dataframes["customers"], "week")
monthly_active = active_customers_over_time(dataframes["customers"], "month")

account_status over time 

In [ ]:
def status_distribution_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.groupBy(*(group_cols + [F.col("account_status")]))
          .agg(F.countDistinct("customer_id").alias("customer_count"))
          .orderBy(*group_cols, "account_status")
    )

daily_status   = status_distribution_over_time(dataframes["customers"], "day")
monthly_status = status_distribution_over_time(dataframes["customers"], "month")